In [ ]:
import os
import sys
from dotenv import load_dotenv

# Clone repository if needed
if not os.path.exists('RAG_TECHNIQUES'):
    os.system('git clone https://github.com/NirDiamant/RAG_TECHNIQUES.git')
else:
    print('RAG_TECHNIQUES already cloned, skipping.')

# Load .env FIRST — before any imports that need the API key
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')

# Verify key loaded correctly
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY not found — check your .env file"
print("API key loaded successfully.")

# Add RAG_TECHNIQUES to path so helper_functions and evaluation modules can be found
sys.path.insert(0, 'RAG_TECHNIQUES')

# Now safe to import modules that depend on the API key
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

from helper_functions import *
from evaluation.evalute_rag import *

print("All imports successful.")

RAG_TECHNIQUES already cloned, skipping.


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader
from langchain_openai import OpenAIEmbeddings

VECTORSTORE_PATH = "vectorstore"

if os.path.exists(VECTORSTORE_PATH):
    # Load from disk if already encoded
    print("Loading existing vectorstore from disk...")
    vectorstore = FAISS.load_local(VECTORSTORE_PATH, OpenAIEmbeddings(), allow_dangerous_deserialization=True)
    print("Vectorstore loaded.")
else:
    # Build from scratch
    print("Building vectorstore from PDFs...")
    
    all_docs = []
    for filename in os.listdir('files'):
        if filename.endswith('.pdf'):
            loader = PyPDFLoader(f'files/{filename}')
            all_docs.extend(loader.load())
    print(f"Loaded {len(all_docs)} pages")

    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    chunks = text_splitter.split_documents(all_docs)
    print(f"Split into {len(chunks)} chunks")

    vectorstore = FAISS.from_documents(chunks, OpenAIEmbeddings())
    vectorstore.save_local(VECTORSTORE_PATH)
    print("Vectorstore built and saved to disk.")

Loading existing vectorstore from disk...
Vectorstore loaded.


In [48]:
llm = ChatOpenAI(model="gpt-4o-mini", max_tokens=1000, temperature=0)

In [49]:
class RetrievalResponse(BaseModel):
    response: str = Field(..., title="Determines if retrieval is necessary", description="Output only 'Yes' or 'No'.")
retrieval_prompt = PromptTemplate(
    input_variables=["query"],
    template="Given the query '{query}', and that you are answering questions about specific documents, should you retrieve context from those documents? Always output 'Yes' unless the question is completely general knowledge unrelated to any document. Output only 'Yes' or 'No'."
)

class RelevanceResponse(BaseModel):
    response: str = Field(..., title="Determines if context is relevant", description="Output only 'Relevant' or 'Irrelevant'.")
relevance_prompt = PromptTemplate(
    input_variables=["query", "context"],
    template="Given the query '{query}' and the context '{context}', determine if the context is relevant. Output only 'Relevant' or 'Irrelevant'."
)

class GenerationResponse(BaseModel):
    response: str = Field(..., title="Generated response", description="The generated response.")
generation_prompt = PromptTemplate(
    input_variables=["query", "context"],
    template="Given the query '{query}' and the context '{context}', generate a response."
)

class SupportResponse(BaseModel):
    response: str = Field(..., title="Determines if response is supported", description="Output 'Fully supported', 'Partially supported', or 'No support'.")
support_prompt = PromptTemplate(
    input_variables=["response", "context"],
    template="Given the response '{response}' and the context '{context}', determine if the response is supported by the context. Output 'Fully supported', 'Partially supported', or 'No support'."
)

class UtilityResponse(BaseModel):
    response: int = Field(..., title="Utility rating", description="Rate the utility of the response from 1 to 5.")
utility_prompt = PromptTemplate(
    input_variables=["query", "response"],
    template="Given the query '{query}' and the response '{response}', rate the utility of the response from 1 to 5."
)

# Create LLMChains for each step
retrieval_chain = retrieval_prompt | llm.with_structured_output(RetrievalResponse)
relevance_chain = relevance_prompt | llm.with_structured_output(RelevanceResponse)
generation_chain = generation_prompt | llm.with_structured_output(GenerationResponse)
support_chain = support_prompt | llm.with_structured_output(SupportResponse)
utility_chain = utility_prompt | llm.with_structured_output(UtilityResponse)

In [72]:
def self_rag(query, vectorstore, top_k=3):
    print(f"\nProcessing query: {query}")
    
    # Step 1: Determine if retrieval is necessary
    print("Step 1: Determining if retrieval is necessary...")
    retrieval_decision = retrieval_chain.invoke({"query": query}).response.strip().lower()
    print(f"Retrieval decision: {retrieval_decision}")
    
    if retrieval_decision == 'yes':
        # Step 2: Retrieve relevant documents
        print("Step 2: Retrieving relevant documents...")
        docs = vectorstore.similarity_search(query, k=top_k)
        sources = [{"file": doc.metadata.get("source", "unknown"), 
                    "page": doc.metadata.get("page", "unknown")} 
                   for doc in docs]
        contexts = [doc.page_content for doc in docs]
        print(f"Retrieved {len(contexts)} documents")
        
        # Step 3: Evaluate relevance of retrieved documents
        print("Step 3: Evaluating relevance of retrieved documents...")
        relevant_contexts = []
        relevant_sources = []
        for i, (context, source) in enumerate(zip(contexts, sources)):
            relevance = relevance_chain.invoke({"query": query, "context": context}).response.strip().lower()
            print(f"Document {i+1} relevance: {relevance}")
            if relevance == 'relevant':
                relevant_contexts.append(context)
                relevant_sources.append(source)
        
        print(f"Number of relevant contexts: {len(relevant_contexts)}")
        
        if not relevant_contexts:
            print("No relevant contexts found.")
            return "There is no information about this in the documents.", []
        
        # Step 4: Generate responses using relevant contexts
        print("Step 4: Generating responses using relevant contexts...")
        responses = []
        for i, (context, source) in enumerate(zip(relevant_contexts, relevant_sources)):
            print(f"Generating response for context {i+1}...")
            response = generation_chain.invoke({"query": query, "context": context}).response
            
            # Step 5: Assess support
            print(f"Step 5: Assessing support for response {i+1}...")
            support = support_chain.invoke({"response": response, "context": context}).response.strip().lower()
            print(f"Support assessment: {support}")
            
            # Step 6: Evaluate utility
            print(f"Step 6: Evaluating utility for response {i+1}...")
            utility = int(utility_chain.invoke({"query": query, "response": response}).response)
            print(f"Utility score: {utility}")
            
            responses.append((response, support, utility, source))
        
        # Select the best response
        print("Selecting the best response...")
        best_response = max(responses, key=lambda x: (x[1] == 'fully supported', x[2]))
        print(f"Best response support: {best_response[1]}, utility: {best_response[2]}")
        return best_response[0], [best_response[3]]
    
    else:
        print("Generating without retrieval...")
        return "There is no information about this in the documents.", []

In [74]:
query = "How old am I?"
response, sources = self_rag(query, vectorstore)

print("\nFinal response:")
print(response)
print("\nSources:")
for source in sources:
    print(f"  {source['file']}, page {source['page']}")


Processing query: How old am I?
Step 1: Determining if retrieval is necessary...
Retrieval decision: no
Generating without retrieval...

Final response:
There is no information about this in the documents.

Sources:
